<a href="https://colab.research.google.com/github/aashishneupanecs24-creator/BIOS-LAB-1BF24CS004/blob/main/Particle_Swarm_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

# ==========================================
# 1. Problem Definition (IEEE 3-Unit System)
# ==========================================
# Cost coefficients: F_i(P_i) = a_i * P_i^2 + b_i * P_i + c_i
a = np.array([0.00156, 0.00194, 0.00482])
b = np.array([7.92, 7.85, 7.97])
c = np.array([561.0, 310.0, 78.0])

# Generation Limits (MW)
P_min = np.array([100.0, 100.0, 50.0])
P_max = np.array([600.0, 400.0, 200.0])

# Total Power Demand (MW)
P_demand = 850.0

# ==========================================
# 2. Objective / Fitness Function
# ==========================================
def objective_function(P):
    """
    Calculates fuel cost + quadratic penalty for violating power demand.
    """
    # Calculate Total Fuel Cost
    fuel_cost = np.sum(a * (P**2) + b * P + c)

    # Penalty for power balance constraint violation (|Sum(P_i) - P_demand|)
    power_mismatch = abs(np.sum(P) - P_demand)
    penalty_weight = 1e5
    penalty = penalty_weight * (power_mismatch ** 2)

    return fuel_cost + penalty

# ==========================================
# 3. Particle Swarm Optimization (PSO)
# ==========================================
def run_pso(num_particles=50, max_iterations=200):
    num_dimensions = len(P_min)  # 3 generators

    # Hyperparameters
    w = 0.7        # Inertia weight
    c1 = 1.5       # Cognitive weight
    c2 = 1.5       # Social weight

    # Initialize particle positions randomly within [P_min, P_max]
    positions = np.random.uniform(
        low=P_min, high=P_max, size=(num_particles, num_dimensions)
    )

    # Initialize velocities randomly within [-10%, +10%] of generation span
    vel_range = 0.1 * (P_max - P_min)
    velocities = np.random.uniform(
        low=-vel_range, high=vel_range, size=(num_particles, num_dimensions)
    )

    # Track Personal Bests
    pbest_positions = np.copy(positions)
    pbest_scores = np.array([objective_function(p) for p in positions])

    # Track Global Best
    gbest_index = np.argmin(pbest_scores)
    gbest_position = np.copy(pbest_positions[gbest_index])
    gbest_score = pbest_scores[gbest_index]

    # Main PSO Optimization Loop
    for it in range(max_iterations):
        for i in range(num_particles):
            # Stochastic variables
            r1 = np.random.rand(num_dimensions)
            r2 = np.random.rand(num_dimensions)

            # Update Velocity
            cognitive = c1 * r1 * (pbest_positions[i] - positions[i])
            social = c2 * r2 * (gbest_position - positions[i])
            velocities[i] = w * velocities[i] + cognitive + social

            # Update Position
            positions[i] += velocities[i]

            # Apply Bound Constraints (Clip to P_min and P_max)
            positions[i] = np.clip(positions[i], P_min, P_max)

            # Evaluate Fitness
            score = objective_function(positions[i])

            # Update Personal Best
            if score < pbest_scores[i]:
                pbest_scores[i] = score
                pbest_positions[i] = np.copy(positions[i])

                # Update Global Best
                if score < gbest_score:
                    gbest_score = score
                    gbest_position = np.copy(positions[i])

    return gbest_position, gbest_score

# ==========================================
# 4. Execution & Results
# ==========================================
if __name__ == "__main__":
    best_P, best_cost = run_pso(num_particles=60, max_iterations=300)

    # Calculate pure fuel cost excluding penalty
    pure_fuel_cost = np.sum(a * (best_P**2) + b * best_P + c)
    total_generated = np.sum(best_P)

    print("----- Optimal Power Generation Dispatch -----")
    for i in range(len(best_P)):
        print(f"Generator {i+1}: {best_P[i]:.3f} MW  (Limits: [{P_min[i]}, {P_max[i]}])")

    print("--------------------------------------------")
    print(f"Total Power Generated : {total_generated:.3f} MW (Target: {P_demand} MW)")
    print(f"Total Minimum Fuel Cost: ${pure_fuel_cost:.2f} / hour")

----- Optimal Power Generation Dispatch -----
Generator 1: 547.957 MW  (Limits: [100.0, 600.0])
Generator 2: 102.043 MW  (Limits: [100.0, 400.0])
Generator 3: 200.000 MW  (Limits: [50.0, 200.0])
--------------------------------------------
Total Power Generated : 850.000 MW (Target: 850.0 MW)
Total Minimum Fuel Cost: $8365.26 / hour
